In [8]:
!pip install mlflow

In [9]:
import mlflow
import mlflow.tensorflow

# Initialisation de MLflow
mlflow.set_experiment("Face_Classification")
mlflow.tensorflow.autolog()

In [10]:
import kagglehub

# Chemins des données
REAL_FACES_DIR = kagglehub.dataset_download("tunguz/70000-real-faces-1")
FAKE_FACES_DIR = kagglehub.dataset_download("tunguz/1-million-fake-faces")

In [11]:
import random
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import EfficientNetB0
import os

# Charger les données
def load_images(directory, label, max_images=None):
    images, labels = [], []
    all_files = [
        os.path.join(root, filename)
        for root, _, files in os.walk(directory)
        for filename in files
        if filename.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]
    if max_images:
        all_files = random.sample(all_files, min(len(all_files), max_images))
    for filepath in all_files:
        try:
            img = load_img(filepath, target_size=(128, 128))
            img_array = img_to_array(img)
            images.append(img_array)
            labels.append(label)
        except Exception as e:
            print(f"Erreur lors du chargement de l'image {filepath}: {e}")
    return images, labels

In [12]:
import numpy as np

def preprocess_data(real_dir, fake_dir, max_images=5000):
    real_images, real_labels = load_images(real_dir, label=0, max_images=max_images)
    fake_images, fake_labels = load_images(fake_dir, label=1, max_images=max_images)
    images = np.array(real_images + fake_images, dtype="float32")
    labels = np.array(real_labels + fake_labels)
    images = preprocess_input(images)
    return images, labels

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

# Création du modèle
def build_model():
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(128, 128, 3))
    base_model.trainable = False
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [14]:
from tensorflow.keras.applications.efficientnet import preprocess_input

# Prédire si l'image est réelle ou fake
def predict_image(image, model):
    """Prédire si l'image est réelle ou fake."""
    image_preprocessed = preprocess_input(np.expand_dims(image, axis=0))
    prediction = model.predict(image_preprocessed)
    predicted_label = "Real" if prediction[0][0] <= 0.5 else "Fake"
    return predicted_label, prediction[0][0]

In [15]:
# Choisir une image réelle aléatoire afin de la prédire
def choose_random_real_image(real_dir, model):
    all_files = [
        os.path.join(root, filename)
        for root, _, files in os.walk(real_dir)
        for filename in files
        if filename.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]
    if not all_files:
        print("Aucune image réelle trouvée.")
        return
    random_image_path = random.choice(all_files)
    img = load_img(random_image_path, target_size=(128, 128))
    img_array = img_to_array(img)

    # Prédiction du modèle
    predicted_label, prediction_score = predict_image(img_array, model)

    img.show()
    print(f"Image réelle choisie : {random_image_path}")
    print(f"Prédiction du modèle pour image réelle : {predicted_label} ({prediction_score:.2f})")

In [16]:
# Choisir une image fake aléatoire afin de la prédire
def choose_random_fake_image(fake_dir, model):
    all_files = [
        os.path.join(root, filename)
        for root, _, files in os.walk(fake_dir)
        for filename in files
        if filename.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]
    if not all_files:
        print("Aucune image fake trouvée.")
        return
    random_image_path = random.choice(all_files)
    img = tensorflow.keras.preprocessing.image.load_img(random_image_path, target_size=(128, 128))
    img_array = tensorflow.keras.preprocessing.image.img_to_array(img)

    # Prédiction du modèle
    predicted_label, prediction_score = predict_image(img_array, model)

    img.show()
    print(f"Image fake choisie : {random_image_path}")
    print(f"Prédiction du modèle pour image fake : {predicted_label} ({prediction_score:.2f})")

In [17]:
from sklearn.model_selection import train_test_split

# Préparer les données
images, labels = preprocess_data(REAL_FACES_DIR, FAKE_FACES_DIR, max_images=5000)
X_train, X_temp, y_train, y_temp = train_test_split(images, labels, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

Erreur lors du chargement de l'image /root/.cache/kagglehub/datasets/tunguz/1-million-fake-faces/versions/3/1m_faces_00_01_02_03/1m_faces_00_01_02_03/1m_faces_00/9G6G661H8A.jpg: image file is truncated (0 bytes not processed)


In [18]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Augmentation des données
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)
train_generator = datagen.flow(X_train, y_train, batch_size=32)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Entraîner le modèle
with mlflow.start_run():
    model = build_model()
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ModelCheckpoint('best_model_efficientnet.keras', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
    ]
    history = model.fit(
        train_generator,
        validation_data=(X_val, y_val),
        epochs=50,
        callbacks=callbacks
    )
    # Débloquer certaines couches pour fine-tuning
    model.layers[0].trainable = True
    for layer in model.layers[0].layers[:len(model.layers[0].layers) // 2]:
        layer.trainable = False

    # Recompiler avec un learning rate réduit
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    history_fine_tune = model.fit(
        train_generator,
        validation_data=(X_val, y_val),
        epochs=20,
        callbacks=callbacks
    )

    # Évaluation
    test_loss, test_accuracy = model.evaluate(X_test, y_test)
    print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
    model.save("fine_tuned_face_classifier_model.keras")

    # Exemple d'utilisation des nouvelles méthodes
    choose_random_real_image(REAL_FACES_DIR, model)
    choose_random_fake_image(FAKE_FACES_DIR, model)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


2025/01/07 14:08:36 WARNING mlflow.tensorflow: Unrecognized dataset type <class 'keras.src.legacy.preprocessing.image.NumpyArrayIterator'>. Dataset logging skipped.


Epoch 1/50


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 780ms/step - accuracy: 0.6345 - loss: 0.6303

...

219/219 ━━━━━━━━━━━━━━━━━━━━ 211s 948ms/step - accuracy: 0.7413 - loss: 0.5189 - val_accuracy: 0.7433 - val_loss: 0.5215 - learning_rate: 0.0010
Epoch 6/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 824ms/step - accuracy: 0.7438 - loss: 0.5276

219/219 ━━━━━━━━━━━━━━━━━━━━ 213s 969ms/step - accuracy: 0.7438 - loss: 0.5276 - val_accuracy: 0.7473 - val_loss: 0.5124 - learning_rate: 0.0010
Epoch 7/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 205s 932ms/step - accuracy: 0.7417 - loss: 0.5156 - val_accuracy: 0.7400 - val_loss: 0.5192 - learning_rate: 0.0010
Epoch 8/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 802ms/step - accuracy: 0.7379 - loss: 0.5174

219/219 ━━━━━━━━━━━━━━━━━━━━ 208s 947ms/step - accuracy: 0.7379 - loss: 0.5174 - val_accuracy: 0.7467 - val_loss: 0.5097 - learning_rate: 0.0010
Epoch 9/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 814ms/step - accuracy: 0.7625 - loss: 0.4994

219/219 ━━━━━━━━━━━━━━━━━━━━ 212s 964ms/step - accuracy: 0.7625 - loss: 0.4995 - val_accuracy: 0.7607 - val_loss: 0.5026 - learning_rate: 0.0010
Epoch 10/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 209s 946ms/step - accuracy: 0.7582 - loss: 0.4964 - val_accuracy: 0.7507 - val_loss: 0.5032 - learning_rate: 0.0010
Epoch 11/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 211s 956ms/step - accuracy: 0.7551 - loss: 0.5067 - val_accuracy: 0.7493 - val_loss: 0.5041 - learning_rate: 0.0010
Epoch 12/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 261s 956ms/step - accuracy: 0.7613 - loss: 0.4901 - val_accuracy: 0.7500 - val_loss: 0.5034 - learning_rate: 0.0010
Epoch 13/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 784ms/step - accuracy: 0.7721 - loss: 0.4787

219/219 ━━━━━━━━━━━━━━━━━━━━ 215s 980ms/step - accuracy: 0.7721 - loss: 0.4786 - val_accuracy: 0.7673 - val_loss: 0.4888 - learning_rate: 2.0000e-04
Epoch 14/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 212s 966ms/step - accuracy: 0.7724 - loss: 0.4642 - val_accuracy: 0.7627 - val_loss: 0.4903 - learning_rate: 2.0000e-04
Epoch 15/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 718ms/step - accuracy: 0.7797 - loss: 0.4639